# Security & Secrets Management — Local Workflow

> **Mission:** Secure a Flask app with database credentials — zero secrets in git, images, or logs. Production-ready secrets handling.
>
> **This notebook:** 100% local secrets workflow (Docker Secrets, .env files, pre-commit hooks). We'll demonstrate bad practices, then fix them with runtime secret injection.
>
> **Supplement notebook:** Azure Key Vault for cloud-native secret management.

---

## Prerequisites

- **Docker Desktop** installed ([download](https://www.docker.com/products/docker-desktop))
- **Git** installed (for pre-commit hooks)
- **Trivy** installed for vulnerability scanning ([docs](https://aquasecurity.github.io/trivy/))

Verify installation:
```bash
docker --version
git --version
trivy --version
```

# Security & Secrets Management — Local Workflow

> **Mission:** Secure a Flask app with database credentials — zero secrets in git, images, or logs. Production-ready secrets handling.
>
> **This notebook:** 100% local secrets workflow (Docker Secrets, .env files, pre-commit hooks). We'll demonstrate bad practices, then fix them with runtime secret injection.
>
> **Supplement notebook:** Azure Key Vault for cloud-native secret management.

---

## Prerequisites

- **Docker Desktop** installed ([download](https://www.docker.com/products/docker-desktop))
- **Git** installed (for pre-commit hooks)
- **Trivy** installed for vulnerability scanning ([docs](https://aquasecurity.github.io/trivy/))

Verify installation:
```bash
docker --version
git --version
trivy --version
```

**Expected output:**
```
...
ENV API_KEY=sk_live_abcdef1234567890
ENV DB_PASSWORD=super_secret_password
...
```

❌ **Problem:** Secrets are permanently embedded in the image. Even if you delete the image and rebuild, the old image layers are still cached. If pushed to Docker Hub, the secrets are public forever.

✅ **Rule:** NEVER use `ENV` for secrets in Dockerfiles.

---

## Cell 2: Good Practice — Environment Variables (Local Dev Only)

**Goal:** Use environment variables passed at runtime instead of build-time secrets.

**What we're doing:**
- Create a Dockerfile **without** hardcoded secrets
- Pass secrets via `-e` flag at `docker run`
- Demonstrate that the image is now clean (no secrets in history)

**Why this works:** Secrets are provided at container startup, not baked into the image. The image can be pushed to a public registry safely.

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
✅ No secrets found in image!
Running container with runtime secrets:
API Key: sk_live_runtime_key
DB Password: runtime_password
App running with runtime secrets...
```

✅ **Improvement:** Secrets are no longer in the image. However, `-e` secrets are visible in `docker inspect` and shell history. For production, use Docker Secrets or Kubernetes Secrets (mounted files).

---

## Cell 3: .env File + Docker Compose (Development)

**Goal:** Use `.env` files for local development secrets (never commit to git).

**What we're doing:**
- Create a `.env` file with secrets
- Create a `docker-compose.yml` that loads `.env`
- Demonstrate that secrets are loaded from the file, not hardcoded

**Why this works:** `.env` is gitignored. Developers can create their own `.env` locally without risking commits. Production uses different secret sources (Key Vault, Secrets Manager).

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
API Key: local_dev_key_123
DB Password: local_dev_password_456
✅ Secrets loaded from .env file!
```

✅ **Best practice for local dev:**
- **Commit** `.env.example` (template with placeholder values)
- **Gitignore** `.env` (actual secrets, never committed)
- **Developers** copy `.env.example` to `.env` and fill in their own secrets
- **Production** uses Docker Secrets, Kubernetes Secrets, or Key Vault (next cells)

---

## Cell 4: Docker Secrets (Swarm Mode)

**Goal:** Use Docker Secrets to mount secrets as files (more secure than environment variables).

**What we're doing:**
- Initialize Docker Swarm (required for Docker Secrets)
- Create a secret with `docker secret create`
- Deploy a service that reads the secret from `/run/secrets/`

**Why this is better:** Secrets are mounted as in-memory files, not environment variables. They never appear in `docker inspect`, process lists, or logs.

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
✅ DB Password loaded from Docker Secret: super_secret_db_password
```

✅ **Production pattern:** Docker Secrets are encrypted at rest, transmitted over TLS, and mounted as in-memory tmpfs files. They never touch disk and are removed when the container stops.

**Note:** Docker Secrets require Swarm mode. For Kubernetes, use Kubernetes Secrets (next cell).

---

## Cell 5: Kubernetes Secrets (Base64 Encoding)

**Goal:** Use Kubernetes Secrets to inject secrets into pods (common in production clusters).

**What we're doing:**
- Create a Kubernetes Secret from a literal value
- Create a pod that loads the secret as an environment variable
- Demonstrate that secrets are base64-encoded (NOT encrypted)

**Why this matters:** K8s Secrets are base64-encoded by default. For production, enable encryption at rest with EncryptionConfiguration.

In [ ]:
# TODO: Implement this cell


**Key takeaways:**
- **Base64 ≠ encryption**: Anyone with `kubectl` access can decode secrets
- **Encryption at rest**: Configure K8s with EncryptionConfiguration to encrypt secrets in etcd
- **RBAC**: Use Role-Based Access Control to restrict who can read secrets
- **Cloud-native alternatives**: Azure Key Vault, AWS Secrets Manager integrate with K8s for true secret management

✅ **Production pattern:** Use Kubernetes External Secrets Operator to sync from Key Vault/Secrets Manager into K8s Secrets.

---

## Cell 6: Scanning Images for Vulnerabilities (Trivy)

**Goal:** Use Trivy to scan Docker images for known vulnerabilities and exposed secrets.

**What we're doing:**
- Scan an image with Trivy (finds CVEs in dependencies)
- Scan for secrets with `trivy fs` (detects hardcoded credentials)
- Demonstrate automated security scanning in CI/CD

**Why this matters:** Vulnerabilities in base images (Python, Node, etc.) and accidentally committed secrets can be caught before deployment.

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
HIGH: AWS Access Key detected
API_KEY=sk_live_abcdef1234567890
```

✅ **CI/CD integration:**
```yaml
# .github/workflows/security.yml
- name: Run Trivy scanner
  uses: aquasecurity/trivy-action@master
  with:
    image-ref: 'myapp:latest'
    format: 'sarif'
    output: 'trivy-results.sarif'
    severity: 'CRITICAL,HIGH'
```

**Why this matters:** Automated scanning catches vulnerabilities and leaked secrets before they reach production. Fail the build if HIGH/CRITICAL issues are found.

---

## Cell 7: Pre-Push Hook for Secret Detection

**Goal:** Install a Git pre-push hook that scans commits for secrets before pushing to remote.

**What we're doing:**
- Create a pre-push hook that searches for secret patterns
- Test it by attempting to commit a file with a hardcoded API key
- Block the push if secrets are detected

**Why this matters:** Prevents developers from accidentally pushing secrets to GitHub. Once secrets are in git history, they're permanently compromised (even if deleted later).

In [ ]:
# TODO: Implement this cell


**Expected output:**
```
Test 1: Pushing clean file...
✅ No secrets detected. Push allowed.

Test 2: Pushing file with secret...
❌ ERROR: Potential secret detected!
   Pattern matched: api_key\s*=\s*['"][^'"]+['"]
   PUSH BLOCKED. Remove secrets and try again.
```

✅ **Enterprise pattern:** Use tools like:
- **pre-commit** ([https://pre-commit.com/](https://pre-commit.com/)) — framework for managing git hooks
- **gitleaks** — specialized secret scanner with 1000+ patterns
- **GitHub Secret Scanning** — automatic scanning on push (free for public repos)

**Critical:** Pre-commit hooks are client-side. For enforcement, enable server-side scanning (GitHub Advanced Security, GitLab Secret Detection).

---

## Cell 8: Secrets Rotation Strategy

**Goal:** Demonstrate a zero-downtime secret rotation workflow.

**What we're doing:**
- Deploy two containers reading from a secret
- Update the secret in the secret store
- Restart containers with graceful rollout (one at a time)
- Verify no downtime during rotation

**Why this matters:** Secrets must be rotated regularly (every 90 days in most compliance frameworks). The rotation must not cause service outages.

In [ ]:
# TODO: Implement this cell


**Why zero downtime is possible:**
- **Rolling updates**: New containers start before old ones stop
- **Dual-password window**: Database accepts both passwords during transition
- **Health checks**: Kubernetes/Swarm waits for new container to be ready before stopping old one

✅ **Best practice:** Automate rotation with:
- **Azure Key Vault**: Automatic rotation policies
- **AWS Secrets Manager**: Lambda-based rotation functions
- **HashiCorp Vault**: Dynamic secrets with TTL (auto-rotation)

**Compliance:** SOC 2, PCI-DSS, HIPAA all require regular secret rotation (typically 90 days).

---

## Cell 9: RBAC — Least Privilege Principle

**Goal:** Demonstrate role-based access control for secrets (who can read what).

**What we're doing:**
- Define RBAC policies for Kubernetes Secrets
- Show how to restrict secret access to specific namespaces/pods
- Demonstrate the principle of least privilege

**Why this matters:** Not all containers should have access to all secrets. A frontend service doesn't need database credentials. RBAC limits blast radius if one service is compromised.

In [ ]:
# TODO: Implement this cell


**Key RBAC Concepts:**

| Concept | Purpose | Example |
|---------|---------|---------|
| **ServiceAccount** | Identity for a pod | `frontend-sa`, `backend-sa` |
| **Role** | Permissions within a namespace | "Can read secret X" |
| **RoleBinding** | Assigns role to ServiceAccount | "frontend-sa has role secret-reader" |
| **ClusterRole** | Permissions across all namespaces | (Use sparingly!) |

✅ **Cloud equivalents:**
- **Azure**: Managed Identity + Key Vault Access Policies
- **AWS**: IAM Roles + Secrets Manager Resource Policies
- **GCP**: Workload Identity + Secret Manager IAM

**Best practice:** Create a ServiceAccount per microservice, grant only required secrets via RBAC.

---

## Cell 10: Audit Logs — Who Accessed What Secret When

**Goal:** Enable audit logging to track secret access (compliance requirement).

**What we're doing:**
- Show how to enable audit logs in Kubernetes
- Demonstrate Azure Key Vault access logs
- Query logs to answer: "Who accessed the database password in the last 30 days?"

**Why this matters:** SOC 2, ISO 27001, HIPAA require audit trails. If a secret is compromised, you need to know who had access.

In [ ]:
# TODO: Implement this cell


**Audit log use cases:**

1. **Compliance audits**: Prove to auditors that only authorized services accessed production secrets
2. **Security incidents**: Determine blast radius of a compromised credential
3. **Access reviews**: Quarterly review of who has access to what secrets
4. **Anomaly detection**: Alert when a secret is accessed from an unusual IP or time

✅ **Enterprise pattern:**
- **Centralized logging**: Forward all audit logs to SIEM (Splunk, Azure Sentinel, AWS Security Hub)
- **Automated alerts**: Trigger alerts on high-risk events (secret accessed from unknown IP)
- **Retention**: Keep logs for 1-7 years (depends on compliance framework)

**Next steps:** Integrate audit logs with your security monitoring stack (Ch.5 Monitoring & Observability).